# Paying per call with x402

An agent without an account pays each call in USDC on Base (about $0.02).
Payments are **off by default**; `max_usd_per_call` is a hard cap: x402 refuses to sign
anything more expensive and the client raises `EconDashPaymentRequired`.

Use a dedicated wallet with a small balance. Never commit the private key.

In [ ]:
%pip install -q "econdash[x402]"

In [ ]:
import os

from eth_account import Account

from econdash import EconDash, EconDashPaymentRequired

signer = Account.from_key(os.environ["AGENT_PRIVATE_KEY"])
ed = EconDash(pay_with_x402=True, x402_signer=signer, max_usd_per_call=0.05)

# Zimbabwe is outside the free trial, so this call is paid
s = ed.get_series("gdp-nominal", "ZWE", start=2015)
print(s.indicator_name_en, [(p.date.year, round(p.value / 1e9, 1)) for p in s.data])

In [ ]:
# What happens without consent: the 402 challenge comes back as an exception, nothing is paid
try:
    EconDash(base_url="https://econdash.org/api/v1/m2m").list_countries()
except EconDashPaymentRequired as e:
    accept = e.challenge["accepts"][0]
    print("price:", int(accept["amount"]) / 1e6, "USDC on", accept["network"])